# 09.12 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/12-agents-tool-calling/exercises.ipynb)

Answers in [solutions.ipynb](solutions.ipynb).

In [ ]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

In [ ]:
sys.path.insert(0, str(repo_file("lib/course_llm.py").parent))
import course_llm
from course_llm import ask, ask_json, chat, info
info();

In [ ]:
sys.path.insert(0, str(repo_file("data/download.py").parent))
from download import load_retail
import re
sales = load_retail()                                   # ShopWave's order lines
docs_dir = repo_file("data/shopwave_docs/returns.md").parent

def order_status(invoice):
    """Look up one order by its 6-digit invoice number."""
    if not re.fullmatch(r"C?\d{6}", invoice.strip()):
        return "Error: an invoice number is 6 digits, for example 536365."
    lines = sales[sales["Invoice"] == invoice.strip()]
    if lines.empty:
        return f"No order found with invoice {invoice}."
    status = "cancelled" if invoice.startswith("C") else "delivered"
    total = (lines["Quantity"] * lines["Price"]).sum()
    return (f"Invoice {invoice}: placed {lines['InvoiceDate'].min():%d %b %Y}, {len(lines)} lines, "
            f"total £{total:,.2f}, shipped to {lines['Country'].iloc[0]}, status {status}.")

In [ ]:
def refund_policy(_=""):
    """ShopWave's returns and refunds rules."""
    return ((docs_dir / "returns.md").read_text() + "\n" + (docs_dir / "refunds.md").read_text())

def product_search(query):
    """Find products whose name contains all the query words; show typical price."""
    words = query.lower().split()
    names = sales["Description"].fillna("").str.lower()
    hits = sales[names.apply(lambda n: all(w in n for w in words))]
    if hits.empty:
        return f"No products match '{query}'."
    top = hits.groupby("Description")["Price"].agg(["median", "count"]).nlargest(5, "count")
    return "; ".join(f"{d.title()} (£{r['median']:.2f})" for d, r in top.iterrows())

TOOL_FUNCTIONS = {"order_status": order_status, "refund_policy": refund_policy,
                  "product_search": product_search}

In [ ]:
import json
AGENT_SYSTEM = """You are ShopWave's support agent. You can use these tools:
- order_status(argument = 6-digit invoice number): date, total, country and status of an order
- refund_policy(argument = ""): ShopWave's returns and refunds rules
- product_search(argument = a few words of a product name): matching products and typical prices
Each turn, reply with JSON: a short thought, the tool to use, and its argument.
When you have enough information, use the tool "final_answer" with the answer to the customer as the argument.
Use only facts from tool results. Never invent order details or prices."""

STEP = {"type": "object", "properties": {
            "thought": {"type": "string"},
            "tool": {"type": "string", "enum": [*TOOL_FUNCTIONS, "final_answer"]},
            "argument": {"type": "string"}},
        "required": ["thought", "tool", "argument"], "additionalProperties": False}

In [ ]:
def run_agent(question, max_steps=5, verbose=True):
    messages = [{"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        reply = chat(messages, system=AGENT_SYSTEM, max_tokens=400, temperature=0, schema=STEP)
        decision = json.loads(reply)
        if verbose:
            print(f"step {step}: {decision['tool']}({decision['argument']!r})  # {decision['thought']}")
        if decision["tool"] == "final_answer":
            return decision["argument"]
        result = TOOL_FUNCTIONS[decision["tool"]](decision["argument"])
        if verbose:
            print(f"        result: {result[:150]}")
        messages += [{"role": "assistant", "content": reply},
                     {"role": "user", "content": f"Tool result: {result}"}]
    return "Sorry, I could not finish this request. A colleague will reply by email."

In [ ]:
def order_status_checked(invoice):
    """Same lookup, but a bad number gets an instruction, not an example to copy."""
    if not re.fullmatch(r"C?\d{6}", invoice.strip()):
        return "No valid invoice number. Ask the customer for their 6-digit invoice number."
    return order_status(invoice)

GUARDED_TOOLS = {**TOOL_FUNCTIONS, "order_status": order_status_checked}
GUARDED_SYSTEM = AGENT_SYSTEM + """
Rules:
- Call each tool at most once with the same argument.
- If you need information the customer has not given (such as an invoice number), use final_answer to ask for it.
- Write the final answer as one or two plain sentences for the customer, never JSON."""

In [ ]:
def run_guarded_agent(question, max_steps=5):
    messages, seen = [{"role": "user", "content": question}], set()
    for step in range(1, max_steps + 1):
        reply = chat(messages, system=GUARDED_SYSTEM, max_tokens=400, temperature=0, schema=STEP)
        d = json.loads(reply)
        print(f"step {step}: {d['tool']}({d['argument']!r})")
        if d["tool"] == "final_answer":
            return d["argument"]
        if (d["tool"], d["argument"]) in seen:                 # guardrail in code, not in the prompt
            result = "You already have this result. Answer the customer now with final_answer."
        else:
            seen.add((d["tool"], d["argument"]))
            result = GUARDED_TOOLS[d["tool"]](d["argument"])
        messages += [{"role": "assistant", "content": reply},
                     {"role": "user", "content": f"Tool result: {result}"}]
    return "Sorry, I could not finish this request. A colleague will reply by email."

## Exercise 1: plural search

`product_search("hot water bottles")` finds nothing, because product names say "bottle". Make the search tolerant: drop a trailing "s" from each query word. Then re-run the hot water bottle question with the guarded agent. Does it need fewer steps?

In [ ]:
# your code here

## Exercise 2: dates belong in code

Write a tool `return_deadline(invoice)` that looks up the order date, adds 30 days and says whether that is before or after a given "today" (use 2011-12-10, the day after the data ends, so results are reproducible). Add it to the tools and the system prompt, then ask *"Order 581587: can I still return it?"* and *"Order 536365: can I still return it?"*.

In [ ]:
# your code here

## Exercise 3: a token budget

Change `run_guarded_agent` so it adds up `course_llm.last_usage` after every step and stops with the apology once the conversation has used more than 3,000 tokens. Which of the four questions are closest to the limit?

In [ ]:
# your code here

## Exercise 4: an action that needs approval

Write a tool `issue_refund(invoice)` that does **not** refund anything: it adds the request to a list `APPROVAL_QUEUE` and tells the model "A colleague will review this refund within 1 working day." Why is this the right design for a support agent? Which other ShopWave actions should work the same way?

In [ ]:
# your code here